# Prerequisite 08 · 强化学习入学测试

[上一章：从多臂老虎机走进 RL](Prerequisite07B_Bandit_Bridge.ipynb) · [预备篇导航](Prerequisite00_Index.ipynb) · [进入 Chapter01](../02_chapters/Chapter01.ipynb)

闭卷完成。不会的题不是失败，而是告诉你应该回到哪个预备章。建议概念题达到 12/15、代码测试全部通过后进入 RL 主线。


## 概念题

1. list、tuple、dict 各适合保存什么？
2. `(B,F) @ (F,A)` 的输出 shape 是什么？
3. `axis=1` 在 `(B,A)` 的 Q 值上表示什么？
4. 移动平均窗口变大有什么代价？
5. 期望和方差分别描述什么？
6. 梯度下降为什么沿负梯度更新？
7. train、validation、test 为什么不能混用？
8. `zero_grad/backward/step` 各做什么？
9. `detach` 和 `no_grad` 的使用场景是什么？
10. `terminated` 和 `truncated` 为什么影响 bootstrap？
11. 为什么用样本平均可以估计期望？样本数变成 4 倍，误差大约变成多少？
12. $\pi(a\mid s)$ 中竖线左右分别是什么？为什么策略梯度要用 log 概率？
13. 为什么神经网络必须有非线性激活？Q 表和 Q 网络有什么区别？
14. `(4,84,84)` 的输入经过 `Conv2d(4,32,8,stride=4)` 后 shape 是多少？为什么 Atari 要堆叠 4 帧？
15. 用一句话解释“新估计 = 旧估计 + 步长 × (目标 − 旧估计)”，并说出 ε=0 的贪心策略可能出什么问题。


<details><summary>参考要点</summary>

1. 可变序列、固定记录、键值配置；2. `(B,A)`；3. 对每个样本的动作维操作；4. 更滞后并可能掩盖突变；5. 中心趋势与离散程度；6. 负梯度是局部下降最快方向；7. 防止选择偏差和泄漏；8. 清梯度、反传、更新；9. 切断局部图与关闭整个块的梯度；10. 真终止没有未来价值，时间截断可能仍有；11. 大数定律，误差约减半（$1/\sqrt n$）；12. 左边是结果（动作），右边是条件（状态）；乘积变求和、避免下溢、便于求导；13. 多层线性仍是线性；Q 表逐格存储、Q 网络用参数近似并能泛化；14. `(32,20,20)`，单帧看不出速度和方向；15. 用误差修正估计；可能过早锁定次优动作、永远不再探索。

</details>


## 代码题 1：NumPy batch 与动作索引


In [1]:
import numpy as np

q = np.array([[1., 4., 2.], [5., 0., 3.], [2., 1., 6.]])
actions = np.array([1, 0, 2])
chosen = q[np.arange(len(actions)), actions]
assert np.allclose(chosen, [4., 5., 6.])
assert np.array_equal(q.argmax(axis=1), actions)
print("NumPy test passed:", chosen)


NumPy test passed: [4. 5. 6.]


## 代码题 2：PyTorch 训练循环


In [2]:
import torch

torch.manual_seed(8)
x = torch.linspace(-1, 1, 80).unsqueeze(1)
y = -1.7*x + .3
model = torch.nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=.15)
first_loss = None
for _ in range(80):
    loss = torch.nn.functional.mse_loss(model(x), y)
    if first_loss is None: first_loss = loss.item()
    optimizer.zero_grad(); loss.backward(); optimizer.step()
assert loss.item() < first_loss*.01
print("PyTorch test passed; loss:", round(loss.item(), 6))


PyTorch test passed; loss: 0.0


## 代码题 3：终止语义


In [3]:
def target(reward, next_value, gamma, terminated):
    return reward + (0.0 if terminated else gamma*next_value)

assert target(1, 10, .9, True) == 1
assert target(1, 10, .9, False) == 10
print("terminal-mask test passed")


terminal-mask test passed


## 代码题 4：CNN 输出 shape 与 Q-learning 更新


In [4]:
import numpy as np
import torch
import torch.nn as nn

conv = nn.Sequential(nn.Conv2d(4, 32, 8, stride=4), nn.ReLU(), nn.Conv2d(32, 64, 4, stride=2), nn.ReLU())
with torch.no_grad():
    out = conv(torch.zeros(2, 4, 84, 84))
assert tuple(out.shape) == (2, 64, 9, 9), out.shape

Q = np.zeros((2, 2)); Q[1] = [0.0, 4.0]
s, a, r, s_next, alpha, gamma = 0, 1, 1.0, 1, 0.5, 0.9
Q[s, a] += alpha * (r + gamma * Q[s_next].max() - Q[s, a])
assert np.isclose(Q[0, 1], 2.3)
print("CNN shape 与 Q-learning 更新测试通过")


CNN shape 与 Q-learning 更新测试通过


## 判定

- 概念 0–8：按顺序完成全部预备篇（包括带 B 的 4 个补课 notebook）。
- 概念 9–11：只补错题对应章节，再重做代码题。
- 概念 12–15 且代码全过：进入 [Chapter01](../02_chapters/Chapter01.ipynb)。

通过后也不用背所有 API。真正重要的是：遇到 shape、梯度和环境问题时，知道怎样检查，而不是把它误认为 RL 算法失效。
